# Convolution backend scaling

Periodic stencil convolution via `JSymmetricTensor @ SymmetricTensor`, compared across **NumPy** and all available Idpy backends (`METAL_T`, `OCL_T`, `CUDA_T`, `CTYPES_T`).

**Device contract (LBM-style):** kernels act on **flat length-`V`** C-contiguous buffers. Tenet ownership lives on arrays (`src.tenet`); logical shape is per-tenet via `set_convolution_shape(shape, tenet=...)`. Host multi-d arrays are packed with `pack_lattice_flat` / `unpack_lattice_flat` (axis-0-fastest, matching LBM stride macros). IdpyMemory stays C-order — never upload multi-d Fortran arrays. `set_active_tenet` is optional soft default only.

**Metrics:** `t_site_ns = t / V` (nanoseconds per lattice site), mean ± SEM over trial medians (see [`OCL_vs_Metal_Bench.ipynb`](OCL_vs_Metal_Bench.ipynb)).

**Modes:**
- **e2e** — NumPy: full `@` each timing; device: `@` with flat field/kernel already on device.
- **compute** — device only: timed `K_ConvolvePeriodic` via `DeployProfiling` (compile excluded by warmup).

**Hardware:** run `IdpyHardware()` below, then set `DEVICE`, `CL_KIND`, and optionally subset `BACKENDS`.

**Stencils:** 1D/2D 5-point centered derivative along axis 0; 3D 7-point Laplacian.

Set `QUICK_RUN = True` for a short smoke sweep before a full run.

Metal uses a buffer **memory pool** (`Tenet.SetMemoryPool`); e2e `@` repeatedly allocates `dst` via the pool (no fresh Metal Buffer each call). Prefer **compute** mode for raw kernel comparison.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../")

import numpy as np
import matplotlib.pyplot as plt

from idpy.Utils.IdpySymbolic import SymmetricTensor, JSymmetricTensor
from idpy.Utils.CustomTypes import CustomTypes
from idpy.Utils.SimpleTiming import SimpleTiming
from idpy.IdpyCode import (
    CUDA_T, OCL_T, METAL_T, CTYPES_T,
    idpy_langs_sys, idpy_langs_human_dict,
    GetTenet, IdpyHardware, IdpyMemory,
)
from idpy.IdpyStencils.IdpyConvolution import (
    K_ConvolvePeriodic,
    set_active_tenet,
    clear_active_tenet,
    clear_idea_cache,
    set_convolution_shape,
    clear_convolution_shape,
    pack_lattice_flat,
    unpack_lattice_flat,
)

In [ ]:
BLOCK = 128
N_WARM = 5
N_INNER = 8
N_TRIALS = 16
DTYPE = np.float32
MAX_RUNTIME_S = None  # optional per-point cap

# --- hardware selection (see IdpyHardware() output) ---
DEVICE = 0
CL_KIND = "gpu"  # OpenCL only; ignored by Metal / CUDA / ctypes

QUICK_RUN = False
SIZES = {
    1: [2 ** k for k in range(10, 25)],
    2: [64, 128, 256, 512, 1024],
    3: [32, 64, 128, 256],
}
if QUICK_RUN:
    SIZES = {1: [2 ** 16], 2: [128], 3: [32]}

DEVICE_LANGS = [l for l in (METAL_T, OCL_T, CUDA_T, CTYPES_T) if idpy_langs_sys.get(l)]
BACKENDS = ["numpy"] + DEVICE_LANGS  # subset e.g. ["numpy", METAL_T]
DIMS = (1, 2, 3)
MODES = ("e2e", "compute")

TOL = 5e-5 if DTYPE == np.float32 else 1e-12

print("Available device backends:", [idpy_langs_human_dict.get(l, l) for l in DEVICE_LANGS])
print("Benchmark backends:", BACKENDS)
print(f"Hardware: DEVICE={DEVICE}, CL_KIND={CL_KIND!r}")
print(f"Stats: {N_TRIALS} trials × median of {N_INNER} (warmup {N_WARM}), dtype={DTYPE}")
for dim in DIMS:
    sizes = SIZES[dim]
    print(f"  dim={dim}: {len(sizes)} sizes, V in [{int(np.prod((sizes[0],)*dim)):,}, {int(np.prod((sizes[-1],)*dim)):,}]")

In [ ]:
from idpy.IdpyStencils.IdpyConvolution import pack_lattice_flat, unpack_lattice_flat


def make_tenet(lang):
    return GetTenet({"lang": lang, "device": DEVICE, "cl_kind": CL_KIND})


def backend_label(backend):
    if backend == "numpy":
        return "numpy"
    return idpy_langs_human_dict.get(backend, backend)


def volume(shape):
    return int(np.prod(shape))


def mean_sem(xs):
    xs = np.asarray(xs, dtype=float)
    n = len(xs)
    mean = float(np.mean(xs))
    sem = float(np.std(xs, ddof=1) / np.sqrt(n)) if n > 1 else 0.0
    return mean, sem


def stable_profile(idea, args, n_warm=N_WARM, n_inner=N_INNER, n_trials=N_TRIALS):
    for _ in range(n_warm):
        idea.DeployProfiling(args)
    trial_medians = []
    for _ in range(n_trials):
        times = [idea.DeployProfiling(args)[1] for _ in range(n_inner)]
        trial_medians.append(float(np.median(times)))
    return mean_sem(trial_medians)


def stable_wall(fn, n_warm=N_WARM, n_inner=N_INNER, n_trials=N_TRIALS):
    for _ in range(n_warm):
        fn()
    trial_medians = []
    st = SimpleTiming()
    for _ in range(n_trials):
        times = []
        for _ in range(n_inner):
            st.Start()
            fn()
            st.End()
            times.append(st.GetElapsedTime()["time_s"])
        trial_medians.append(float(np.median(times)))
    return mean_sem(trial_medians)


def ns_per_site(t_s, V):
    return (t_s / V) * 1e9 if V > 0 else float("nan")


def ns_per_site_sem(t_sem, V):
    return (t_sem / V) * 1e9 if V > 0 else float("nan")


def fmt_pm(mean, sem, prec=4):
    return f"{mean:.{prec}g}±{sem:.{prec}g}"


def print_table(rows, headers):
    widths = [
        max(len(str(h)), max((len(f"{r[i]:.4g}" if isinstance(r[i], float) else str(r[i])) for r in rows), default=0))
        for i, h in enumerate(headers)
    ]
    fmt = "  ".join(f"{{:{w}}}" for w in widths)
    print(fmt.format(*headers))
    print(fmt.format(*["-" * w for w in widths]))
    for r in rows:
        cells = []
        for x in r:
            cells.append(f"{x:.4g}" if isinstance(x, float) else str(x))
        print(fmt.format(*cells))


def plot_scaling(xs, series, xlabel, ylabel, title, xscale="log", yscale="log", yerr=None):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for label, ys in series.items():
        err = yerr.get(label) if yerr else None
        ax.errorbar(xs, ys, yerr=err, marker="o", label=label, capsize=3)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


def make_kernel(dim, dtype=DTYPE):
    if dim == 1:
        c = np.zeros(5, dtype=dtype)
        c[:] = np.array([1.0 / 12, -2.0 / 3, 0.0, 2.0 / 3, -1.0 / 12], dtype=dtype)
        return c
    if dim == 2:
        c = np.zeros((5, 5), dtype=dtype)
        c[:, 2] = np.array([1.0 / 12, -2.0 / 3, 0.0, 2.0 / 3, -1.0 / 12], dtype=dtype)
        return c
    if dim == 3:
        c = np.zeros((3, 3, 3), dtype=dtype)
        c[1, 1, 1] = -6.0
        c[0, 1, 1] = c[2, 1, 1] = 1.0
        c[1, 0, 1] = c[1, 2, 1] = 1.0
        c[1, 1, 0] = c[1, 1, 2] = 1.0
        return c
    raise ValueError(f"unsupported dim={dim}")


def kernel_taps(kernel):
    center = tuple(s // 2 for s in kernel.shape)
    nz = np.nonzero(kernel)
    offsets = np.array(list(zip(*nz)), dtype=np.int64) - np.array(center)
    coeffs = kernel[nz]
    return offsets, coeffs


def make_field(shape, rng, dtype=DTYPE):
    return np.ascontiguousarray(rng.standard_normal(shape).astype(dtype))


def convolve_numpy(kernel, field):
    d = field.ndim
    st = JSymmetricTensor(
        d=d, rank=0, ranks=[0, 0],
        c_dict={0: np.array(kernel, copy=True)},
        dtype=kernel.dtype,
    )
    fld = SymmetricTensor(
        d=d, rank=0,
        c_dict={0: np.array(field, copy=True)},
        dtype=field.dtype,
    )
    return np.asarray((st @ fld)[0])


def setup_device_matmul(kernel, field, shape, tenet):
    """Flat LBM-style device tensors for JSymmetricTensor @."""
    set_convolution_shape(shape, tenet=tenet)
    k_dev = IdpyMemory.OnDevice(np.ascontiguousarray(kernel), tenet=tenet)
    f_flat = IdpyMemory.OnDevice(pack_lattice_flat(field), tenet=tenet)
    d = len(shape)
    st = JSymmetricTensor(
        d=d, rank=0, ranks=[0, 0],
        c_dict={0: k_dev}, dtype=kernel.dtype,
    )
    fld = SymmetricTensor(d=d, rank=0, c_dict={0: f_flat}, dtype=field.dtype)
    return st, fld


def device_output_numpy(st, fld, shape):
    out = st @ fld
    val = out[0]
    flat = np.asarray(val.D2H() if hasattr(val, "D2H") else val)
    return unpack_lattice_flat(flat, shape)


def bench_numpy_e2e(kernel, field):
    return stable_wall(lambda: convolve_numpy(kernel, field))


def bench_device_e2e(st, fld, shape):
    return stable_wall(lambda: device_output_numpy(st, fld, shape))


def bench_device_compute(kernel, field, shape, tenet):
    offsets, coeffs = kernel_taps(kernel)
    V = volume(shape)
    f_name = "float" if kernel.dtype == np.float32 else "double"
    custom_types = CustomTypes({"FType": f_name, "SType": "int"}).Push()
    kern = K_ConvolvePeriodic(offsets, coeffs, shape, custom_types=custom_types)
    grid = ((V + BLOCK - 1) // BLOCK, 1, 1)
    block = (BLOCK, 1, 1)
    idea = kern(tenet=tenet, grid=grid, block=block)
    dst = IdpyMemory.Zeros(V, dtype=kernel.dtype, tenet=tenet)
    src_flat = IdpyMemory.OnDevice(pack_lattice_flat(field), tenet=tenet)
    return stable_profile(idea, [dst, src_flat])

## Hardware

`IdpyHardware()` lists languages and devices. Selection knobs are in the config cell: `DEVICE`, `CL_KIND`, and `BACKENDS`.

In [ ]:
IdpyHardware()

## Correctness gate

Small shapes per dimension: each device backend must match NumPy within tolerance (flat LBM pack/unpack path) before timing.

In [ ]:
CORRECT_SHAPES = {1: (64,), 2: (32, 32), 3: (16, 16, 16)}
rng = np.random.default_rng(0)
correctness_ok = True

for dim in DIMS:
    shape = CORRECT_SHAPES[dim]
    kernel = make_kernel(dim)
    field = make_field(shape, rng)
    ref = convolve_numpy(kernel, field)
    print(f"dim={dim} shape={shape} ref ok, max|ref|={np.max(np.abs(ref)):.4g}")

    for lang in DEVICE_LANGS:
        if lang not in BACKENDS:
            continue
        label = backend_label(lang)
        tenet = make_tenet(lang)
        clear_idea_cache()
        try:
            st, fld = setup_device_matmul(kernel, field, shape, tenet)
            out = device_output_numpy(st, fld, shape)
            err = float(np.max(np.abs(out - ref)))
            status = "OK" if err <= TOL else "FAIL"
            print(f"  {label} @: max|Δ|={err:.3e} [{status}]")
            if err > TOL:
                correctness_ok = False
        except Exception as exc:
            correctness_ok = False
            print(f"  {label} @: ERROR {exc}")
        finally:
            clear_convolution_shape(tenet=tenet)
            clear_idea_cache()
            tenet.End()

if not correctness_ok:
    raise RuntimeError("Correctness gate failed — fix before benchmarking")
print("Correctness gate passed.")

## Benchmark sweep

Loops over dimension, grid size, backend, and mode. OOM / backend errors are logged and skipped.

In [ ]:
import time

results = []
numpy_e2e_by_dim = {dim: {} for dim in DIMS}

for dim in DIMS:
    kernel = make_kernel(dim)
    for N in SIZES[dim]:
        shape = (N,) * dim
        V = volume(shape)
        nbytes = V * np.dtype(DTYPE).itemsize
        print(f"\n=== dim={dim} shape={shape} V={V:,} (~{nbytes/1e6:.1f} MB field) ===")

        field = make_field(shape, np.random.default_rng(N))

        if "numpy" in BACKENDS:
            try:
                t0 = time.perf_counter()
                t, t_sem = bench_numpy_e2e(kernel, field)
                if MAX_RUNTIME_S and (time.perf_counter() - t0) > MAX_RUNTIME_S:
                    print("  numpy e2e: skipped (MAX_RUNTIME_S)")
                else:
                    t_site = ns_per_site(t, V)
                    t_site_sem = ns_per_site_sem(t_sem, V)
                    numpy_e2e_by_dim[dim][V] = t
                    results.append({
                        "dim": dim, "N": N, "shape": shape, "V": V,
                        "backend": "numpy", "mode": "e2e",
                        "t": t, "t_sem": t_sem,
                        "t_site_ns": t_site, "t_site_ns_sem": t_site_sem,
                        "speedup_vs_numpy": 1.0,
                    })
                    print(
                        f"  numpy e2e: {t*1e3:.3f}±{t_sem*1e3:.3f} ms  "
                        f"{fmt_pm(t_site, t_site_sem)} ns/site"
                    )
            except Exception as exc:
                print(f"  numpy e2e: ERROR {exc}")

        for lang in DEVICE_LANGS:
            if lang not in BACKENDS:
                continue
            label = backend_label(lang)
            tenet = make_tenet(lang)
            clear_idea_cache()
            try:
                st, fld = setup_device_matmul(kernel, field, shape, tenet)

                if "e2e" in MODES:
                    try:
                        t0 = time.perf_counter()
                        t, t_sem = bench_device_e2e(st, fld, shape)
                        if MAX_RUNTIME_S and (time.perf_counter() - t0) > MAX_RUNTIME_S:
                            print(f"  {label} e2e: skipped (MAX_RUNTIME_S)")
                        else:
                            t_site = ns_per_site(t, V)
                            t_site_sem = ns_per_site_sem(t_sem, V)
                            ref_t = numpy_e2e_by_dim[dim].get(V)
                            speedup = (ref_t / t) if ref_t and t > 0 else float("nan")
                            results.append({
                                "dim": dim, "N": N, "shape": shape, "V": V,
                                "backend": lang, "mode": "e2e",
                                "t": t, "t_sem": t_sem,
                                "t_site_ns": t_site, "t_site_ns_sem": t_site_sem,
                                "speedup_vs_numpy": speedup,
                            })
                            sp = f" speedup={speedup:.2f}x" if np.isfinite(speedup) else ""
                            print(
                                f"  {label} e2e: {t*1e3:.3f}±{t_sem*1e3:.3f} ms  "
                                f"{fmt_pm(t_site, t_site_sem)} ns/site{sp}"
                            )
                    except Exception as exc:
                        print(f"  {label} e2e: ERROR {exc}")

                if "compute" in MODES:
                    try:
                        clear_idea_cache()
                        t0 = time.perf_counter()
                        t, t_sem = bench_device_compute(kernel, field, shape, tenet)
                        if MAX_RUNTIME_S and (time.perf_counter() - t0) > MAX_RUNTIME_S:
                            print(f"  {label} compute: skipped (MAX_RUNTIME_S)")
                        else:
                            t_site = ns_per_site(t, V)
                            t_site_sem = ns_per_site_sem(t_sem, V)
                            ref_t = numpy_e2e_by_dim[dim].get(V)
                            speedup = (ref_t / t) if ref_t and t > 0 else float("nan")
                            results.append({
                                "dim": dim, "N": N, "shape": shape, "V": V,
                                "backend": lang, "mode": "compute",
                                "t": t, "t_sem": t_sem,
                                "t_site_ns": t_site, "t_site_ns_sem": t_site_sem,
                                "speedup_vs_numpy": speedup,
                            })
                            sp = f" speedup={speedup:.2f}x" if np.isfinite(speedup) else ""
                            print(
                                f"  {label} compute: {t*1e3:.3f}±{t_sem*1e3:.3f} ms  "
                                f"{fmt_pm(t_site, t_site_sem)} ns/site{sp}"
                            )
                    except Exception as exc:
                        print(f"  {label} compute: ERROR {exc}")
            finally:
                clear_convolution_shape(tenet=tenet)
                clear_idea_cache()
                tenet.End()

print(f"\nCollected {len(results)} benchmark points.")

## Results tables

Per dimension and mode: ms and ns/site (mean ± SEM).

In [ ]:
for dim in DIMS:
    for mode in MODES:
        rows = []
        subset = [r for r in results if r["dim"] == dim and r["mode"] == mode]
        if not subset:
            continue
        Vs = sorted({r["V"] for r in subset})
        backends_seen = []
        for r in subset:
            bl = backend_label(r["backend"]) if r["backend"] != "numpy" else "numpy"
            if bl not in backends_seen:
                backends_seen.append(bl)
        for V in Vs:
            row = [V]
            for bl in backends_seen:
                match = [
                    r for r in subset
                    if r["V"] == V and (
                        (r["backend"] == "numpy" and bl == "numpy")
                        or backend_label(r["backend"]) == bl
                    )
                ]
                if match:
                    r = match[0]
                    row += [
                        fmt_pm(r["t"] * 1e3, r["t_sem"] * 1e3),
                        fmt_pm(r["t_site_ns"], r["t_site_ns_sem"]),
                    ]
                else:
                    row += ["—", "—"]
            rows.append(row)
        headers = ["V"] + [f"{bl}:{h}" for bl in backends_seen for h in ("ms", "ns/site")]
        print(f"\ndim={dim} mode={mode}")
        print_table(rows, headers)

## Scaling plots

`ns/site` vs volume `V` (log-log), one panel per dimension and mode.

In [ ]:
for dim in DIMS:
    for mode in MODES:
        subset = [r for r in results if r["dim"] == dim and r["mode"] == mode]
        if not subset:
            continue
        Vs = sorted({r["V"] for r in subset})
        labels = sorted({
            backend_label(r["backend"]) if r["backend"] != "numpy" else "numpy"
            for r in subset
        })
        series, yerr = {}, {}
        for label in labels:
            ys, es = [], []
            for V in Vs:
                match = [
                    x for x in subset
                    if x["V"] == V and (
                        (x["backend"] == "numpy" and label == "numpy")
                        or backend_label(x["backend"]) == label
                    )
                ]
                if match:
                    ys.append(match[0]["t_site_ns"])
                    es.append(match[0]["t_site_ns_sem"])
            if ys:
                series[label] = ys
                yerr[label] = es
        if series:
            plot_scaling(
                Vs, series,
                xlabel="V (sites)",
                ylabel="ns / site",
                title=f"Convolution dim={dim} mode={mode}",
                yerr=yerr,
            )


## Speedup vs NumPy

Ratio `t_numpy_e2e / t_backend` (linear y).

In [ ]:
for dim in DIMS:
    for mode in ("e2e", "compute"):
        subset = [
            r for r in results
            if r["dim"] == dim and r["mode"] == mode and r["backend"] != "numpy"
        ]
        if not subset:
            continue
        labels = sorted({backend_label(r["backend"]) for r in subset})
        series, yerr = {}, {}
        all_vs = sorted({r["V"] for r in subset})
        for label in labels:
            ys = []
            es = []
            xs = []
            for V in all_vs:
                match = [
                    r for r in subset
                    if r["V"] == V and backend_label(r["backend"]) == label
                ]
                if match and np.isfinite(match[0]["speedup_vs_numpy"]):
                    xs.append(V)
                    ys.append(match[0]["speedup_vs_numpy"])
                    es.append(0.0)
            if ys:
                series[label] = ys
                yerr[label] = es
        if not series:
            continue
        # use union xs: plot each series with its own x via errorbar in helper
        fig, ax = plt.subplots(figsize=(6.5, 4))
        for label in labels:
            if label not in series:
                continue
            xs = [
                r["V"] for r in subset
                if backend_label(r["backend"]) == label and np.isfinite(r["speedup_vs_numpy"])
            ]
            ys = [
                r["speedup_vs_numpy"] for r in subset
                if backend_label(r["backend"]) == label and np.isfinite(r["speedup_vs_numpy"])
            ]
            ax.plot(xs, ys, marker="o", label=label)
        ax.set_xscale("log")
        ax.set_xlabel("V (sites)")
        ax.set_ylabel("speedup vs numpy e2e")
        ax.set_title(f"Speedup dim={dim} mode={mode}")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()
        fig.tight_layout()
        plt.show()


## How to read the results

- **Falling `ns/site` with `V`**: launch / Python overhead amortized; small grids look artificially slow per site.
- **Flat `ns/site`**: saturated regime — fair backend comparison.
- **e2e vs compute**: gap shows `@` / tap-extract overhead vs raw `K_ConvolvePeriodic`.
- **Device layout**: flat length-`V` + `pack_lattice_flat` (LBM axis-0-fastest); IdpyMemory remains C-contiguous.
- **Metal**: `SetMemoryPool` message is expected; zero-copy `.host` can make transfers look near-free vs OpenCL (see [`OCL_vs_Metal_Bench.ipynb`](OCL_vs_Metal_Bench.ipynb)).
- **dtype**: default `float32` on device; prefer large `V` when comparing GPU backends.